# Module 13 — Notebook 2 Solutions: False Positive and False Negative Analysis

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_contains, check_length

# Load data
data_path = Path("../../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(trigger in response for trigger in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

print("Setup complete.")

## Exercise 1 — Solution: Extract False Negatives

In [ ]:
fn_examples = [
    record
    for record, pred, label in zip(outputs, predictions_v1, ground_truth)
    if not pred and label
]

fn_count = len(fn_examples)
fn_ids = [r['id'] for r in fn_examples]

print(f"False negatives: {fn_count}")
print(f"FN IDs: {fn_ids}")
for r in fn_examples:
    print(f"  {r['id']} ({r['model']}): {r['response']!r}")

In [ ]:
check_type(fn_examples, list, "fn_examples is a list")
check_equal(fn_count, 2, "FN count is 2")
check_contains(fn_ids, 'out_011', "fn_ids contains out_011")
check_contains(fn_ids, 'out_015', "fn_ids contains out_015")

## Exercise 2 — Solution: Count False Positives

In [ ]:
fp_count = sum(1 for p, l in zip(predictions_v1, ground_truth) if p and not l)
print(f"False positives: {fp_count}")

In [ ]:
check_equal(fp_count, 0, "classifier v1 has no false positives")

## Exercise 3 — Solution: Which Model Produced the FNs?

In [ ]:
fn_models = [r['model'] for r in fn_examples]
all_fn_same_model = len(set(fn_models)) == 1

print(f"FN model names: {fn_models}")
print(f"All FNs from the same model: {all_fn_same_model}")

In [ ]:
check_equal(all_fn_same_model, True, "All FNs came from the same model")
check_equal(fn_models[0], 'model-b-v1', "FN model is model-b-v1")